# 1. Environment and asset catalogue

**Purpose:** Load the shared libraries and table names, remove obsolete Databricks widgets, and build the Pearl 700 aircraft-to-engine catalogue used by the rest of the notebook.

**Inputs:** The production asset-management table and the fixed `Pearl 700` engine type.

**Output:** `asset_df`, `asset_rows`, and the sorted aircraft choices. All identifiers are normalized before they reach a widget or table filter.

**Run order:** Run this cell first. It requires the Databricks `spark` and `dbutils` objects.


In [ ]:
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

ASSET_TABLE = "ehm_fleetstore_prd1eun82719_internal.assetmanagement.aircraftengine"
MASTER_TABLE = "ehm_fleetstore_prd1eun82719_internal.`g700-pearl700`.`emucontinuousmaster-aircraftengine-cda`"
SCAN_TABLE = "ehm_fleetstore_prd1eun82719_internal.`g700-pearl700`.`emucontinuousscan-cda`"
ENGINE_TYPE_CODE = "Pearl 700"
CHANNELS = ("AC", "BC")
PARAMETER_PREFIX = "Parameter:EVHMU_EEC_"

# Remove legacy Databricks widgets once; linked ipywidgets in the next cell replace them completely.
for legacy_widget_name in (
    "AircraftID",
    "EngineSerialNumber",
    "Flight_Date",
    "num_years",
    "FlightStartDateTime",
):
    try:
        dbutils.widgets.remove(legacy_widget_name)
    except Exception:
        pass

# Normalize identifiers to strings so UI values and table filters use the same representation.
asset_df = (
    spark.table(ASSET_TABLE)
    .where(F.col("EngineTypeCode") == ENGINE_TYPE_CODE)
    .select(
        F.col("LatestAircraftLatestIdentifier").cast("string").alias("AircraftIdentifier"),
        F.col("LatestAircraftId").alias("AircraftId"),
        F.col("EngineSerialNumber").cast("string").alias("EngineSerialNumber"),
        F.col("EngineId").alias("EngineId"),
        F.col("LatestOperatorName").alias("OperatorName"),
    )
    .where(
        F.col("AircraftIdentifier").isNotNull()
        & F.col("AircraftId").isNotNull()
        & F.col("EngineSerialNumber").isNotNull()
    )
    .dropDuplicates()
)

asset_rows = asset_df.collect()
aircraft_options = sorted({row.AircraftIdentifier for row in asset_rows})
if not aircraft_options:
    raise RuntimeError(f"No aircraft were found for engine type {ENGINE_TYPE_CODE}.")

print(f"Loaded {len(aircraft_options)} aircraft with {ENGINE_TYPE_CODE} engines.")

# 2. Aircraft, engine, and flight selection

**Purpose:** Select one aircraft, one engine, and a calendar range, then load every matching flight window in that range.

**Controls:** Changing the aircraft updates the engine list in place. Choose the dates and press **Load selected flights**. The initial full-notebook run loads the displayed defaults automatically.

**Output:** The selected identifiers, normalized flight windows, master records, and a Spark scan DataFrame named `cda`. The scan is restricted to the selected engine and joined to every selected flight window.

**Important:** After changing a selection, run the next cell and all cells below it so materialized data, plots, correlations, and diagnostics use the new revision.


In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display as ipy_display

# Build validated aircraft and engine maps once so changing an aircraft only updates widget options.
aircraft_ids_by_identifier = {}
engine_ids_by_aircraft = {}
for row in asset_rows:
    aircraft_ids_by_identifier.setdefault(row.AircraftIdentifier, set()).add(row.AircraftId)
    aircraft_engines = engine_ids_by_aircraft.setdefault(row.AircraftIdentifier, {})
    esn = str(row.EngineSerialNumber)
    if esn in aircraft_engines and aircraft_engines[esn] != row.EngineId:
        raise ValueError(f"Engine serial number {esn} maps to more than one engine ID.")
    aircraft_engines[esn] = row.EngineId

for aircraft_identifier, internal_ids in aircraft_ids_by_identifier.items():
    if len(internal_ids) != 1:
        raise ValueError(
            f"Aircraft {aircraft_identifier} maps to {len(internal_ids)} current internal aircraft IDs."
        )

existing_aircraft_widget = globals().get("aircraft_widget")
existing_engine_widget = globals().get("engine_widget")
existing_start_widget = globals().get("start_date_widget")
existing_end_widget = globals().get("end_date_widget")

previous_aircraft = getattr(existing_aircraft_widget, "value", None)
previous_engine = getattr(existing_engine_widget, "value", None)
default_aircraft = previous_aircraft if previous_aircraft in aircraft_options else aircraft_options[0]
default_engine_options = sorted(engine_ids_by_aircraft[default_aircraft])
default_engine = previous_engine if previous_engine in default_engine_options else default_engine_options[0]

# Preserve an existing UI range; otherwise open on the latest seven-day period containing data.
previous_start_date = getattr(existing_start_widget, "value", None)
previous_end_date = getattr(existing_end_widget, "value", None)
if previous_start_date is None or previous_end_date is None:
    latest_calendar_row = (
        spark.table(MASTER_TABLE)
        .where(
            (F.col("AircraftIdentifier").cast("string") == default_aircraft)
            & (F.col("EngineSerialNumber").cast("string") == default_engine)
        )
        .agg(F.max("CalendarId").alias("LatestCalendarId"))
        .first()
    )
    latest_calendar_id = latest_calendar_row.LatestCalendarId if latest_calendar_row else None
    previous_end_date = (
        datetime.strptime(str(int(latest_calendar_id)), "%Y%m%d").date()
        if latest_calendar_id is not None
        else date.today()
    )
    previous_start_date = previous_end_date - timedelta(days=6)

aircraft_widget = widgets.Dropdown(
    options=aircraft_options,
    value=default_aircraft,
    description="Aircraft:",
    layout=widgets.Layout(width="420px"),
)
engine_widget = widgets.Dropdown(
    options=default_engine_options,
    value=default_engine,
    description="Engine:",
    layout=widgets.Layout(width="420px"),
)
start_date_widget = widgets.DatePicker(
    value=previous_start_date,
    description="Start date:",
    layout=widgets.Layout(width="300px"),
)
end_date_widget = widgets.DatePicker(
    value=previous_end_date,
    description="End date:",
    layout=widgets.Layout(width="300px"),
)
load_flights_btn = widgets.Button(
    description="Load selected flights",
    button_style="primary",
    layout=widgets.Layout(width="220px"),
)
selection_output = widgets.Output()

# Update the engine choices in place; no widget is removed or recreated.
def _on_aircraft_change(change):
    new_aircraft = change["new"]
    new_engine_options = sorted(engine_ids_by_aircraft[new_aircraft])
    current_engine = engine_widget.value
    engine_widget.options = new_engine_options
    engine_widget.value = (
        current_engine if current_engine in new_engine_options else new_engine_options[0]
    )


def _month_partitions_between_dates(start_date, end_date):
    current = date(start_date.year, start_date.month, 1)
    final = date(end_date.year, end_date.month, 1)
    partitions = []
    while current <= final:
        partitions.append((current.year, current.month))
        if current.month == 12:
            current = date(current.year + 1, 1, 1)
        else:
            current = date(current.year, current.month + 1, 1)
    return partitions


# Load every selected-engine flight whose master record starts inside the chosen date range.
def _load_selected_flights(_, raise_errors=False):
    load_flights_btn.disabled = True
    try:
        with selection_output:
            clear_output(wait=True)
            chosen_aircraft = aircraft_widget.value
            chosen_engine = engine_widget.value
            chosen_start_date = start_date_widget.value
            chosen_end_date = end_date_widget.value

            if chosen_start_date is None or chosen_end_date is None:
                raise ValueError("Both start and end dates are required.")
            if chosen_start_date > chosen_end_date:
                raise ValueError("Start date must not be later than end date.")

            internal_ids = aircraft_ids_by_identifier[chosen_aircraft]
            chosen_internal_aircraft_id = next(iter(internal_ids))
            chosen_engine_id = engine_ids_by_aircraft[chosen_aircraft][chosen_engine]
            start_calendar_id = int(chosen_start_date.strftime("%Y%m%d"))
            end_calendar_id = int(chosen_end_date.strftime("%Y%m%d"))

            master_df = (
                spark.table(MASTER_TABLE)
                .where(
                    (F.col("AircraftIdentifier").cast("string") == chosen_aircraft)
                    & (F.col("EngineSerialNumber").cast("string") == chosen_engine)
                    & F.col("CalendarId").between(start_calendar_id, end_calendar_id)
                )
            )
            windows_df = (
                master_df.select("StartDatetime", "EndDatetime")
                .where(F.col("StartDatetime").isNotNull() & F.col("EndDatetime").isNotNull())
                .groupBy("StartDatetime")
                .agg(F.max("EndDatetime").alias("EndDatetime"))
                .where(F.col("EndDatetime") >= F.col("StartDatetime"))
            )
            chosen_flight_rows = windows_df.orderBy(F.col("StartDatetime")).collect()

            chosen_scan_table = spark.table(SCAN_TABLE)
            if chosen_flight_rows:
                range_start = min(row.StartDatetime for row in chosen_flight_rows)
                range_end = max(row.EndDatetime for row in chosen_flight_rows)
                partition_filter = F.lit(False)
                for partition_year, partition_month in _month_partitions_between_dates(
                    range_start.date(),
                    range_end.date(),
                ):
                    partition_filter = partition_filter | (
                        (F.col("year") == partition_year)
                        & (F.col("Month") == partition_month)
                    )

                scan_alias = chosen_scan_table.where(
                    (F.col("AircraftId") == chosen_internal_aircraft_id)
                    & (F.col("AssetIdentifier").cast("string") == chosen_engine)
                    & partition_filter
                ).alias("scan")
                window_alias = F.broadcast(windows_df).alias("window")
                join_condition = (
                    (scan_alias["StartDatetime"] == window_alias["StartDatetime"])
                    & scan_alias["Timestamp"].between(
                        window_alias["StartDatetime"],
                        window_alias["EndDatetime"],
                    )
                )
                chosen_cda = (
                    scan_alias.join(window_alias, join_condition, "inner")
                    .select(*[scan_alias[column] for column in chosen_scan_table.columns])
                )
                total_flight_duration = sum(
                    (row.EndDatetime - row.StartDatetime for row in chosen_flight_rows),
                    timedelta(0),
                )
            else:
                range_start = None
                range_end = None
                total_flight_duration = timedelta(0)
                chosen_cda = chosen_scan_table.limit(0)

            # Commit the new state only after every validation and query-planning step succeeds.
            globals().update(
                {
                    "selected_acid": chosen_aircraft,
                    "selected_esn": chosen_engine,
                    "internal_acid": chosen_internal_aircraft_id,
                    "engineid": chosen_engine_id,
                    "window_start_date": chosen_start_date,
                    "window_end_date": chosen_end_date,
                    "start_calendar_id": start_calendar_id,
                    "end_calendar_id": end_calendar_id,
                    "cda_master_dt": master_df,
                    "flight_windows_df": windows_df,
                    "flight_rows": chosen_flight_rows,
                    "flight_available": bool(chosen_flight_rows),
                    "selected_start_dt": range_start,
                    "selected_end_dt": range_end,
                    "start_dt": range_start,
                    "end_dt": range_end,
                    "flight_duration": total_flight_duration,
                    "scan_table": chosen_scan_table,
                    "cda_cols": chosen_scan_table.columns,
                    "cda": chosen_cda,
                    "selection_revision": globals().get("selection_revision", 0) + 1,
                }
            )

            print(
                f"Aircraft: {chosen_aircraft} | Internal aircraft ID: "
                f"{chosen_internal_aircraft_id} | ESN: {chosen_engine} | Engine ID: {chosen_engine_id}"
            )
            print(f"Date range: {chosen_start_date} through {chosen_end_date}")
            print(
                f"Flights loaded: {len(chosen_flight_rows)} | "
                f"Combined flight duration: {total_flight_duration}"
            )
            if chosen_flight_rows:
                print("Run Cell 3 and the cells below it to refresh data, plots, and correlations.")
            else:
                print("No matching flights were found for this selection.")
    except Exception as exc:
        with selection_output:
            print(f"Unable to load the selected flights: {exc}")
        if raise_errors:
            raise
    finally:
        load_flights_btn.disabled = False


aircraft_widget.observe(_on_aircraft_change, names="value")
load_flights_btn.on_click(_load_selected_flights)

selection_controls = widgets.VBox(
    [
        widgets.HBox([aircraft_widget, engine_widget]),
        widgets.HBox([start_date_widget, end_date_widget, load_flights_btn]),
        selection_output,
    ]
)
ipy_display(selection_controls)

# Load the defaults during a full notebook run; later changes use the button above.
_load_selected_flights(None, raise_errors=True)

# 3. Analysis columns and Pandas materialization

**Purpose:** Select the required identifiers, numeric `FlightPhase`, thrust-reverser signals, and contextual shaft-speed/altitude signals from the filtered Spark data.

**Behavior:** Optional AC/BC signals missing from a schema revision are reported and skipped. `FlightPhase` is mandatory and resolved case-insensitively. Timestamp fields are normalized, and invalid phase values become missing values for neutral plot shading.

**Safety:** Conversion to Pandas stops above 1,000,000 rows. Reduce the date range if that guard is reached.

**Output:** `select_para_df`, `_data_df`, `PARAMETER_COLUMNS`, and `FLIGHT_PHASE_COLUMN`.


In [ ]:
# Define channel-neutral suffixes once, then generate matching AC and BC columns programmatically.
THRUST_REVERSER_SUFFIXES = [
    "IAcThrustReverser_tcmIntlV_data",
    "IAcThrustReverser_trLeftDoorSwLocked_LOWER__data",
    "IAcThrustReverser_trLeftDoorSwLocked_UPPER__data",
    "IAcThrustReverser_trLwrDoorLeftSwLocked_data",
    "IAcThrustReverser_trLwrDoorRightSwLocked_data",
    "IAcThrustReverser_trLwrDoorRightSwLocked_flt",
    "IAcThrustReverser_trRightDoorSwLocked_LOWER__data",
    "IAcThrustReverser_trRightDoorSwLocked_UPPER__data",
    "IAcThrustReverser_trUprDoorLeftSwLocked_data",
    "IAcThrustReverser_trUprDoorLeftSwLocked_flt",
    "IAcThrustReverser_trUprDoorRightSwLocked_data",
    "IAcThrustReverser_trUprDoorRightSwLocked_flt",
    "IOSThrustReverser_mainTestEnabled_data",
    "IOSThrustReverser_trDoorPosVa_data",
    "IOSThrustReverser_trDoorPosVb_data",
    "IOSThrustReverser_trDoorPosVex_data",
    "IOSThrustReverser_trDoorPosVex_extflt",
    "IOSThrustReverser_trDoorPosVex_intflt",
    "IOSThrustReverser_trLoSwRaw_data",
    "IOSThrustReverser_trUpSwRaw_data",
    "IOtherProcThrustReverser_trLoLVTSelected_data",
    "IOtherProcThrustReverser_trUpLVTSelected_data",
    "IThrustReverser_loOwnTRLVTRaw_data",
    "IThrustReverser_loOwnTRLVTRngFlt_data",
    "IThrustReverser_trAnomaly_data",
    "IThrustReverser_trArmed_data",
    "IThrustReverser_trDeploySelected_data",
    "IThrustReverser_trDoorDeplStatus_data",
    "IThrustReverser_trDoorPosLost_data",
    "IThrustReverser_trDoorStowStatus_data",
    "IThrustReverser_trInTransit_data",
    "IThrustReverser_trInadvDeploy_data",
    "IThrustReverser_trIsDeployed_data",
    "IThrustReverser_trIsUnlocked_data",
    "IThrustReverser_trJam_data",
    "IThrustReverser_trLVTPosVCtrl_data",
    "IThrustReverser_trLegalDeployCmd_data",
    "IThrustReverser_trLessDeplPos_data",
    "IThrustReverser_trLoLVTSelected_data",
    "IThrustReverser_trLoLVTSelected_flt",
    "IThrustReverser_trMTESTimeExpired_data",
    "IThrustReverser_trMTESV_data",
    "IThrustReverser_trMoreDeplPos_data",
    "IThrustReverser_trUnavailable_data",
    "IThrustReverser_trUpLVTSelected_data",
    "IThrustReverser_trUpLVTSelected_flt",
    "IThrustReverser_trVPrSwVFlt_data",
    "IThrustReverser_trVPrSwV_data",
    "IThrustReverser_upOwnTRLVTRaw_data",
    "IThrustReverser_upOwnTRLVTRngFlt_data",
]

THRUST_REVERSER_COLUMNS = [
    f"{PARAMETER_PREFIX}{channel}_{suffix}"
    for channel in CHANNELS
    for suffix in THRUST_REVERSER_SUFFIXES
]
CONTEXT_COLUMNS = [
    f"{PARAMETER_PREFIX}AC_IHPShaft_nhV_data",
    f"{PARAMETER_PREFIX}BC_IHPShaft_nhV_data",
    f"{PARAMETER_PREFIX}AC_IAircraftState_altitudeC_data",
    f"{PARAMETER_PREFIX}BC_IAircraftState_altitudeC_data",
]
BASE_IDENTIFIER_COLUMNS = [
    "Timestamp",
    "StartDatetime",
    "ParentAssetIdentifier",
    "AssetIdentifier",
]

# Resolve FlightPhase case-insensitively so a harmless schema case change does not break plotting.
scan_schema = set(cda_cols)
phase_matches = [column for column in cda_cols if column.lower() == "flightphase"]
if not phase_matches:
    phase_matches = [column for column in cda_cols if "flightphase" in column.lower()]
if len(phase_matches) != 1:
    raise ValueError(
        "Expected exactly one numeric FlightPhase column; "
        f"found {len(phase_matches)} candidates: {phase_matches}"
    )
FLIGHT_PHASE_COLUMN = phase_matches[0]
IDENTIFIER_COLUMNS = BASE_IDENTIFIER_COLUMNS + [FLIGHT_PHASE_COLUMN]

# Treat identifiers and FlightPhase as mandatory while allowing optional signals to be absent.
missing_identifiers = [column for column in IDENTIFIER_COLUMNS if column not in scan_schema]
if missing_identifiers:
    raise ValueError(f"The scan table is missing required columns: {missing_identifiers}")

requested_parameter_columns = THRUST_REVERSER_COLUMNS + CONTEXT_COLUMNS
unavailable_parameter_columns = [
    column for column in requested_parameter_columns if column not in scan_schema
]
PARAMETER_COLUMNS = [column for column in requested_parameter_columns if column in scan_schema]
selected_cols = list(dict.fromkeys(IDENTIFIER_COLUMNS + PARAMETER_COLUMNS))

if unavailable_parameter_columns:
    print(f"Unavailable optional parameters: {len(unavailable_parameter_columns)}")

# Materialize only the chosen engine and columns after Spark has applied every flight filter.
select_para_df = cda.select(*selected_cols).orderBy(F.col("Timestamp"))
MAX_PANDAS_ROWS = 1_000_000
selected_row_count = select_para_df.count()
if selected_row_count > MAX_PANDAS_ROWS:
    raise RuntimeError(
        f"The selected range contains {selected_row_count:,} samples. "
        f"Reduce the date range below {MAX_PANDAS_ROWS:,} samples before converting to Pandas."
    )
_data_df = select_para_df.toPandas()

if not _data_df.empty:
    _data_df["Timestamp"] = pd.to_datetime(_data_df["Timestamp"], errors="coerce")
    _data_df["StartDatetime"] = pd.to_datetime(_data_df["StartDatetime"], errors="coerce")
    # FlightPhase is numeric by definition; invalid values become missing and receive neutral shading.
    _data_df[FLIGHT_PHASE_COLUMN] = pd.to_numeric(
        _data_df[FLIGHT_PHASE_COLUMN], errors="coerce"
    )
    _data_df = _data_df.sort_values("Timestamp", kind="stable").reset_index(drop=True)

print(f"Loaded samples: {len(_data_df):,}")
print(f"Loaded parameter columns: {len(PARAMETER_COLUMNS)}")
print(f"Flight-phase column: {FLIGHT_PHASE_COLUMN}")


# 4. Input-feature selection

**Purpose:** Choose the signals used by both the plot and the correlation analysis while showing readable names without the common `Parameter:EVHMU_EEC_` prefix.

**Controls:** Search for a parameter to add it, highlight one or more entries to remove them, and press **Apply inputs** to confirm the current set. AC and BC versions remain separate selectable features.

**Output:** `selected_parameters`, a live list shared by the plotting and correlation callbacks. The default selection contains available shaft-speed and altitude context signals.


In [ ]:
import ipywidgets as widgets
from IPython.display import display as ipy_display

# Keep short names in the UI while retaining an unambiguous mapping to the Spark columns.
def _short_parameter_name(column):
    return column[len(PARAMETER_PREFIX):] if column.startswith(PARAMETER_PREFIX) else column


combined_channel_columns = [_short_parameter_name(column) for column in PARAMETER_COLUMNS]
FEATURE_COLUMNS = [
    _short_parameter_name(column)
    for column in CONTEXT_COLUMNS
    if column in PARAMETER_COLUMNS
]

input_search = widgets.Combobox(
    placeholder="Search and add a parameter",
    options=[option for option in combined_channel_columns if option not in FEATURE_COLUMNS],
    description="Add input:",
    ensure_option=True,
    layout=widgets.Layout(width="650px"),
)
input_display = widgets.SelectMultiple(
    options=tuple(FEATURE_COLUMNS),
    description="Plot inputs:",
    layout=widgets.Layout(width="650px", height="280px"),
)
remove_input_btn = widgets.Button(
    description="Remove highlighted",
    button_style="danger",
    icon="trash",
)
apply_input_btn = widgets.Button(
    description="Apply inputs",
    button_style="info",
)
selected_output = widgets.Output()

# Mutate this list in place so plotting callbacks always see the latest applied inputs.
selected_parameters = list(FEATURE_COLUMNS)


def _refresh_search_options():
    active = set(input_display.options)
    input_search.options = [
        option for option in combined_channel_columns if option not in active
    ]


def _sync_selected_parameters():
    selected_parameters[:] = list(input_display.options)


def add_to_input(change):
    value = change["new"]
    if value and value in combined_channel_columns and value not in input_display.options:
        input_display.options = tuple(input_display.options) + (value,)
        _sync_selected_parameters()
        input_search.value = ""
        _refresh_search_options()


def remove_input(_):
    highlighted = set(input_display.value)
    input_display.options = tuple(
        option for option in input_display.options if option not in highlighted
    )
    _sync_selected_parameters()
    _refresh_search_options()


def apply_inputs(_):
    _sync_selected_parameters()
    with selected_output:
        selected_output.clear_output()
        print(f"Applied {len(selected_parameters)} plot inputs.")


input_search.observe(add_to_input, names="value")
remove_input_btn.on_click(remove_input)
apply_input_btn.on_click(apply_inputs)

input_column = widgets.VBox(
    [input_search, input_display, remove_input_btn, apply_input_btn, selected_output]
)
ipy_display(input_column)

# 5. Multi-flight signal plot

**Purpose:** Compare selected AC/BC signals across all flights in the active range without allocating most of the chart to overnight or multi-day gaps.

**X-axis:** Labels are true calendar timestamps. Time remains proportional inside each flight; inter-flight gaps are compressed and marked by a zigzag axis break with the omitted duration. Manual time boundaries snap to the nearest recorded sample.

**Missing data:** A red dotted connector joins the samples on either side of a gap. It is only a visual continuity cue and is not interpolated or measured data. An in-flight gap is detected when the interval exceeds the larger of three median sample periods or five seconds.

**Flight phases:** Numeric `FlightPhase` runs are drawn as low-opacity background bands. Phases 1-6 have fixed colors, unexpected numeric values receive a deterministic color, and missing phase values are grey.


In [ ]:
from collections import OrderedDict

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import clear_output, display as ipy_display
from ipywidgets import SelectionRangeSlider
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.transforms import blended_transform_factory

# Reuse the current applied inputs; this fallback also gives a useful message after partial execution.
selected_parameters = globals().get("selected_parameters", [])

# Reapply the ESN filter defensively in case _data_df was supplied by an earlier notebook run.
data_df = _data_df.copy()
if "AssetIdentifier" in data_df.columns:
    data_df = data_df.loc[
        data_df["AssetIdentifier"].astype("string") == str(selected_esn)
    ].copy()
if "Timestamp" in data_df.columns:
    data_df["Timestamp"] = pd.to_datetime(data_df["Timestamp"], errors="coerce")
if "StartDatetime" in data_df.columns:
    data_df["StartDatetime"] = pd.to_datetime(data_df["StartDatetime"], errors="coerce")
if FLIGHT_PHASE_COLUMN in data_df.columns:
    data_df[FLIGHT_PHASE_COLUMN] = pd.to_numeric(
        data_df[FLIGHT_PHASE_COLUMN], errors="coerce"
    )
data_df = data_df.dropna(subset=["Timestamp", "StartDatetime"]).sort_values(
    ["Timestamp", "StartDatetime"], kind="stable"
)

MAX_SLIDER_POINTS = 1000
MAX_SIGNAL_GROUPS = 16
CHANNEL_COLORS = {"AC": "#1f77b4", "BC": "#ff7f0e"}
KNOWN_PHASE_COLORS = {
    "Phase 1": "#4e79a7",
    "Phase 2": "#59a14f",
    "Phase 3": "#f28e2b",
    "Phase 4": "#e15759",
    "Phase 5": "#b07aa1",
    "Phase 6": "#76b7b2",
    "Unknown": "#bab0ac",
}
PHASE_ALPHA = 0.12
NO_DATA_COLOR = "#c62828"


def _full_parameter_name(short_name):
    return short_name if short_name.startswith(PARAMETER_PREFIX) else PARAMETER_PREFIX + short_name


def _channel_and_signal(short_name):
    for channel in CHANNELS:
        prefix = f"{channel}_"
        if short_name.startswith(prefix):
            return channel, short_name[len(prefix):]
    return "", short_name


def _numeric_series(series):
    non_null = series.dropna()
    if non_null.empty:
        return pd.Series(np.nan, index=series.index, dtype="float64"), False
    is_boolean = non_null.map(lambda value: isinstance(value, (bool, np.bool_))).all()
    if is_boolean:
        return series.map({True: 1.0, False: 0.0}), True
    numeric = pd.to_numeric(series, errors="coerce")
    values = numeric.dropna().to_numpy(dtype=float)
    is_discrete = bool(
        len(values)
        and len(np.unique(values)) <= 8
        and np.allclose(values, np.round(values))
    )
    return numeric, is_discrete


def _match_timestamp_timezone(value, reference):
    timestamp = pd.Timestamp(value)
    if reference.tzinfo is None and timestamp.tzinfo is not None:
        return timestamp.tz_localize(None)
    if reference.tzinfo is not None and timestamp.tzinfo is None:
        return timestamp.tz_localize(reference.tzinfo)
    if reference.tzinfo is not None and timestamp.tzinfo is not None:
        return timestamp.tz_convert(reference.tzinfo)
    return timestamp


def _format_gap_duration(total_seconds):
    total_seconds = max(0, int(round(total_seconds)))
    days, remainder = divmod(total_seconds, 86400)
    hours, remainder = divmod(remainder, 3600)
    minutes, seconds = divmod(remainder, 60)
    if days:
        return f"{days}d {hours:02d}h"
    if hours:
        return f"{hours}h {minutes:02d}m"
    if minutes:
        return f"{minutes}m {seconds:02d}s"
    return f"{seconds}s"


def _build_compressed_timeline(frame):
    """Map each flight to a continuous local timeline and compress only inter-flight gaps."""
    flight_frames = []
    for flight_key, flight_frame in frame.groupby("StartDatetime", sort=False, dropna=False):
        ordered = flight_frame.sort_values("Timestamp", kind="stable").copy()
        actual_start = ordered["Timestamp"].iloc[0]
        actual_end = ordered["Timestamp"].iloc[-1]
        positive_deltas = (
            ordered["Timestamp"].drop_duplicates().sort_values().diff().dt.total_seconds()
        )
        positive_deltas = positive_deltas.loc[positive_deltas > 0]
        cadence_seconds = float(positive_deltas.median()) if not positive_deltas.empty else 1.0
        flight_frames.append(
            {
                "key": flight_key,
                "frame": ordered,
                "actual_start": actual_start,
                "actual_end": actual_end,
                "cadence_seconds": max(cadence_seconds, 0.001),
                "duration_seconds": max(
                    float((actual_end - actual_start).total_seconds()),
                    max(cadence_seconds, 1.0),
                ),
            }
        )

    flight_frames.sort(key=lambda item: item["actual_start"])
    active_seconds = sum(item["duration_seconds"] for item in flight_frames)
    gap_count = max(0, len(flight_frames) - 1)
    visual_gap_seconds = (
        max(active_seconds * min(0.02, 0.15 / gap_count), 1.0)
        if gap_count
        else 0.0
    )

    cursor = 0.0
    timeline_parts = []
    flights = []
    gaps = []
    previous = None
    for item in flight_frames:
        if previous is not None:
            gap_start = cursor
            cursor += visual_gap_seconds
            gaps.append(
                {
                    "plot_start": gap_start,
                    "plot_end": cursor,
                    "plot_center": (gap_start + cursor) / 2.0,
                    "actual_seconds": max(
                        0.0,
                        float((item["actual_start"] - previous["actual_end"]).total_seconds()),
                    ),
                }
            )

        plot_start = cursor
        part = item["frame"].copy()
        part["_plot_x"] = plot_start + (
            part["Timestamp"] - item["actual_start"]
        ).dt.total_seconds()
        plot_end = plot_start + item["duration_seconds"]
        part["_flight_order"] = len(flights)
        timeline_parts.append(part)
        flights.append(
            {
                **item,
                "plot_start": plot_start,
                "plot_end": plot_end,
            }
        )
        cursor = plot_end
        previous = item

    timeline = pd.concat(timeline_parts, ignore_index=True) if timeline_parts else frame.iloc[0:0].copy()
    return timeline, flights, gaps, cursor


def _phase_label(value):
    if pd.isna(value):
        return "Unknown"
    numeric_value = float(value)
    if numeric_value.is_integer():
        return f"Phase {int(numeric_value)}"
    return f"Phase {numeric_value:g}"


def _phase_color(label):
    if label in KNOWN_PHASE_COLORS:
        return KNOWN_PHASE_COLORS[label]
    numeric_text = label.replace("Phase ", "", 1)
    try:
        palette_index = int(abs(float(numeric_text)) * 997) % 20
    except ValueError:
        palette_index = sum(ord(character) for character in label) % 20
    return plt.get_cmap("tab20")(palette_index)


def _build_phase_spans(timeline, flights):
    """Return one background interval for each contiguous numeric FlightPhase run."""
    spans = []
    for flight_order, flight in enumerate(flights):
        # Flight order is assigned during timeline construction and is immune to timestamp formatting.
        flight_frame = timeline.loc[timeline["_flight_order"] == flight_order].copy()
        flight_frame = flight_frame.sort_values("_plot_x", kind="stable")
        if flight_frame.empty:
            continue
        labels = flight_frame[FLIGHT_PHASE_COLUMN].map(_phase_label)
        run_ids = labels.ne(labels.shift()).cumsum()
        runs = list(flight_frame.assign(_phase_label=labels, _phase_run=run_ids).groupby("_phase_run"))
        for run_index, (_, run_frame) in enumerate(runs):
            start_x = float(run_frame["_plot_x"].iloc[0])
            if run_index + 1 < len(runs):
                end_x = float(runs[run_index + 1][1]["_plot_x"].iloc[0])
            else:
                end_x = float(flight["plot_end"])
            spans.append((start_x, max(start_x, end_x), run_frame["_phase_label"].iloc[0]))
    return spans


def _timeline_ticks(flights, maximum_ticks=14):
    """Create readable ticks whose labels remain true calendar timestamps."""
    if not flights:
        return [], []
    if len(flights) > 7:
        selected_indices = np.unique(
            np.linspace(0, len(flights) - 1, num=min(maximum_ticks, len(flights)), dtype=int)
        )
        tick_specs = [(flights[index], 0.0) for index in selected_indices]
    else:
        ticks_per_flight = max(1, maximum_ticks // len(flights))
        tick_specs = []
        for flight in flights:
            fractions = np.linspace(0.0, 1.0, num=ticks_per_flight, endpoint=True)
            for fraction in fractions:
                tick_specs.append((flight, float(fraction)))

    tick_positions = []
    tick_labels = []
    for flight, fraction in tick_specs:
        elapsed_seconds = flight["duration_seconds"] * fraction
        actual_seconds = max(0.0, float((flight["actual_end"] - flight["actual_start"]).total_seconds()))
        actual_time = flight["actual_start"] + pd.to_timedelta(actual_seconds * fraction, unit="s")
        tick_positions.append(flight["plot_start"] + elapsed_seconds)
        tick_labels.append(actual_time.strftime("%d %b %Y\n%H:%M:%S"))
    return tick_positions, tick_labels


def _draw_axis_break(axis, gap, show_label=False):
    """Draw a visible zigzag where a long inter-flight interval was compressed."""
    transform = blended_transform_factory(axis.transData, axis.transAxes)
    half_width = max((gap["plot_end"] - gap["plot_start"]) * 0.32, 0.25)
    center = gap["plot_center"]
    x_values = np.linspace(center - half_width, center + half_width, 7)
    y_values = np.array([-0.018, 0.018, -0.018, 0.018, -0.018, 0.018, -0.018])
    axis.plot(
        x_values,
        y_values,
        color="#333333",
        linewidth=1.3,
        transform=transform,
        clip_on=False,
        zorder=8,
    )
    axis.axvline(center, color="#777777", linewidth=0.7, linestyle=":", alpha=0.45, zorder=1)
    if show_label:
        axis.annotate(
            f"Gap\n{_format_gap_duration(gap['actual_seconds'])}",
            xy=(center, 0),
            xycoords=transform,
            xytext=(0, -42),
            textcoords="offset points",
            ha="center",
            va="top",
            fontsize=8,
            color="#444444",
            annotation_clip=False,
        )


def _plot_parameters(start_time, end_time):
    chosen = list(selected_parameters)
    if not chosen:
        print("No plot inputs are applied. Add at least one parameter in the previous cell.")
        return

    grouped = OrderedDict()
    for short_name in chosen:
        channel, signal = _channel_and_signal(short_name)
        grouped.setdefault(signal, []).append((channel, short_name))

    if len(grouped) > MAX_SIGNAL_GROUPS:
        print(
            f"The current selection creates {len(grouped)} signal groups. "
            f"Reduce it to {MAX_SIGNAL_GROUPS} or fewer for a readable plot."
        )
        return

    visible = data_df.loc[
        data_df["Timestamp"].between(start_time, end_time, inclusive="both")
    ].copy()
    if visible.empty:
        print("No samples fall inside the requested time range.")
        return

    timeline, flights, gaps, timeline_end = _build_compressed_timeline(visible)
    phase_spans = _build_phase_spans(timeline, flights)
    if timeline.empty:
        print("No valid flight timestamps fall inside the requested time range.")
        return

    figure_height = max(5.5, 3.0 * len(grouped))
    figure, axes = plt.subplots(
        len(grouped),
        1,
        figsize=(22, figure_height),
        sharex=True,
        squeeze=False,
    )
    skipped = []

    # Use identical low-opacity phase shading on every axis so signals remain legible.
    for axis in axes[:, 0]:
        for span_start, span_end, phase_name in phase_spans:
            axis.axvspan(
                span_start,
                span_end,
                facecolor=_phase_color(phase_name),
                alpha=PHASE_ALPHA,
                linewidth=0,
                zorder=0,
            )

    # Put AC and BC versions of the same signal on one axis for direct channel comparison.
    for axis, (signal, members) in zip(axes[:, 0], grouped.items()):
        plotted = False
        for channel, short_name in members:
            full_name = _full_parameter_name(short_name)
            if full_name not in timeline.columns:
                skipped.append(short_name)
                continue

            parameter_frame = timeline[
                ["Timestamp", "StartDatetime", "_plot_x", "_flight_order", full_name]
            ].copy()
            parameter_frame["_value"], is_discrete = _numeric_series(parameter_frame[full_name])
            parameter_frame = parameter_frame.dropna(subset=["_value"])
            if parameter_frame.empty:
                skipped.append(short_name)
                continue

            channel_label_pending = True
            previous_flight_endpoint = None
            for flight_order, flight_segment in parameter_frame.groupby("_flight_order", sort=True):
                flight_segment = flight_segment.sort_values("Timestamp", kind="stable")
                positive_deltas = flight_segment["Timestamp"].diff().dt.total_seconds()
                # Use the flight's full sampling cadence so a sparsely populated signal cannot hide a gap.
                median_cadence = flights[int(flight_order)]["cadence_seconds"]
                gap_threshold = max(3.0 * median_cadence, 5.0)
                chunk_ids = positive_deltas.gt(gap_threshold).cumsum()
                chunks = [chunk for _, chunk in flight_segment.groupby(chunk_ids, sort=True)]

                # A red dotted connector marks missing samples; it is deliberately not interpolation.
                if previous_flight_endpoint is not None:
                    first_row = chunks[0].iloc[0]
                    axis.plot(
                        [previous_flight_endpoint[0], first_row["_plot_x"]],
                        [previous_flight_endpoint[1], first_row["_value"]],
                        color=NO_DATA_COLOR,
                        linewidth=1.2,
                        linestyle=(0, (2, 3)),
                        zorder=3,
                    )

                for chunk_index, chunk in enumerate(chunks):
                    if chunk_index:
                        previous_row = chunks[chunk_index - 1].iloc[-1]
                        next_row = chunk.iloc[0]
                        axis.plot(
                            [previous_row["_plot_x"], next_row["_plot_x"]],
                            [previous_row["_value"], next_row["_value"]],
                            color=NO_DATA_COLOR,
                            linewidth=1.2,
                            linestyle=(0, (2, 3)),
                            zorder=3,
                        )
                    axis.plot(
                        chunk["_plot_x"],
                        chunk["_value"],
                        color=CHANNEL_COLORS.get(channel, "#333333"),
                        linewidth=1.25,
                        drawstyle="steps-post" if is_discrete else "default",
                        label=(channel or short_name) if channel_label_pending else "_nolegend_",
                        zorder=4,
                    )
                    channel_label_pending = False

                last_row = chunks[-1].iloc[-1]
                previous_flight_endpoint = (last_row["_plot_x"], last_row["_value"])
                plotted = True

        axis.set_ylabel(signal, fontsize=9)
        axis.grid(True, alpha=0.22, zorder=1)
        axis.margins(y=0.08)
        # The bottom axis receives its break marks later together with duration labels.
        if axis is not axes[-1, 0]:
            for gap in gaps:
                _draw_axis_break(axis, gap, show_label=False)
        if plotted:
            axis.legend(loc="upper right", ncol=max(1, len(members)), framealpha=0.85)
        else:
            axis.text(0.5, 0.5, "No numeric samples", ha="center", va="center", transform=axis.transAxes)

    bottom_axis = axes[-1, 0]
    tick_positions, tick_labels = _timeline_ticks(flights)
    bottom_axis.set_xticks(tick_positions)
    bottom_axis.set_xticklabels(tick_labels, rotation=25, ha="right", fontsize=8)
    bottom_axis.set_xlabel(
        "Flight time - calendar timestamps shown; inter-flight gaps compressed",
        labelpad=48 if gaps else 12,
    )
    for gap in gaps:
        _draw_axis_break(bottom_axis, gap, show_label=True)
    for axis in axes[:, 0]:
        axis.set_xlim(0.0, max(timeline_end, 1.0))

    present_phase_names = sorted(
        {phase_name for _, _, phase_name in phase_spans},
        key=lambda label: (label == "Unknown", label),
    )
    figure_handles = [
        Line2D(
            [0],
            [0],
            color=NO_DATA_COLOR,
            linewidth=1.5,
            linestyle=(0, (2, 3)),
            label="No recorded data",
        )
    ]
    figure_handles.extend(
        Patch(
            facecolor=_phase_color(phase_name),
            alpha=0.30,
            edgecolor="none",
            label=phase_name,
        )
        for phase_name in present_phase_names
    )
    figure.legend(
        handles=figure_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.012),
        ncol=min(4, len(figure_handles)),
        frameon=False,
    )
    figure.suptitle(
        f"Aircraft {selected_acid} | Engine {selected_esn} | "
        f"{len(flights)} flights | {start_time} through {end_time}",
        fontsize=14,
    )
    figure.text(
        0.5,
        0.065,
        "Calendar timestamps are preserved in labels. Axis-break marks compress inter-flight "
        "no-data intervals. Red dotted connectors are not measured data.",
        ha="center",
        va="bottom",
        fontsize=9,
        color="#444444",
    )
    figure.tight_layout(rect=(0.02, 0.105, 0.99, 0.95))
    plt.show()
    plt.close(figure)

    if skipped:
        print("Skipped parameters without numeric samples: " + ", ".join(sorted(set(skipped))))


if data_df.empty:
    print("No samples are available for the selected flights and engine.")
else:
    all_timestamps = pd.DatetimeIndex(data_df["Timestamp"].drop_duplicates().sort_values())
    if len(all_timestamps) > MAX_SLIDER_POINTS:
        indices = np.linspace(
            0,
            len(all_timestamps) - 1,
            num=MAX_SLIDER_POINTS,
            dtype=int,
        )
        slider_timestamps = [all_timestamps[index] for index in np.unique(indices)]
    else:
        slider_timestamps = list(all_timestamps)

    def _timestamp_label(timestamp):
        return pd.Timestamp(timestamp).strftime("%Y-%m-%d %H:%M:%S.%f").rstrip("0").rstrip(".")


    def _nearest_recorded_timestamp(timestamp):
        matched = _match_timestamp_timezone(timestamp, all_timestamps[0])
        nearest_index = all_timestamps.get_indexer([matched], method="nearest")[0]
        return pd.Timestamp(all_timestamps[nearest_index])


    time_slider = SelectionRangeSlider(
        options=[(_timestamp_label(timestamp), timestamp) for timestamp in slider_timestamps],
        index=(0, len(slider_timestamps) - 1),
        description="Time range:",
        orientation="horizontal",
        layout=widgets.Layout(width="1200px"),
        continuous_update=False,
    )
    txt_start = widgets.Text(
        value=_timestamp_label(all_timestamps[0]),
        description="Start:",
        layout=widgets.Layout(width="500px"),
    )
    txt_end = widgets.Text(
        value=_timestamp_label(all_timestamps[-1]),
        description="End:",
        layout=widgets.Layout(width="500px"),
    )
    plot_btn = widgets.Button(
        description="Plot",
        button_style="success",
        icon="line-chart",
        layout=widgets.Layout(width="150px"),
    )
    plot_output = widgets.Output()

    def _render_range(start_time, end_time):
        with plot_output:
            clear_output(wait=True)
            if start_time > end_time:
                start_time, end_time = end_time, start_time
            _plot_parameters(start_time, end_time)


    def _on_plot_click(_):
        try:
            start_time = _nearest_recorded_timestamp(txt_start.value)
            end_time = _nearest_recorded_timestamp(txt_end.value)
        except Exception:
            with plot_output:
                clear_output(wait=True)
                print("Start and end must be valid date-time values.")
            return
        if start_time > end_time:
            start_time, end_time = end_time, start_time
        # Show the exact observations used after a manually entered boundary is snapped.
        txt_start.value = _timestamp_label(start_time)
        txt_end.value = _timestamp_label(end_time)
        _render_range(start_time, end_time)


    def _on_slider_change(change):
        start_time, end_time = map(pd.Timestamp, change["new"])
        txt_start.value = _timestamp_label(start_time)
        txt_end.value = _timestamp_label(end_time)
        _render_range(start_time, end_time)


    plot_btn.on_click(_on_plot_click)
    time_slider.observe(_on_slider_change, names="value")

    controls = widgets.VBox([txt_start, txt_end, time_slider, plot_btn])
    ipy_display(widgets.VBox([controls, plot_output]))
    _render_range(all_timestamps[0], all_timestamps[-1])


# 6. All-parameter correlation screening

**Purpose:** Calculate Pearson correlation between each applied input feature and every eligible numeric or Boolean engine parameter in the selected flights.

**Exclusions:** The input itself and obvious identifiers such as timestamps, aircraft IDs, engine IDs, airline/operator IDs, serial numbers, asset identifiers, and calendar IDs are omitted.

**Controls:** Limit displayed results, cap sampled rows, and require a minimum number of paired observations. Computation runs in Spark batches to handle thousands of candidate columns.

**Output:** Each input gets a full-width, scrollable two-column table. Positive values are ordered from highest to lowest; negative values are ordered from lowest (closest to -1) upward. Every metric keeps its explicit sign, and the common parameter prefix is hidden only in the display.

**Interpretation:** Correlation is a screening statistic and does not establish causation.


In [ ]:
import html
import math

import ipywidgets as widgets
from IPython.display import HTML, clear_output, display as ipy_display
from pyspark.sql.types import BooleanType, NumericType

CORRELATION_BATCH_SIZE = 100
CORRELATION_MAX_INPUTS = 8
CORRELATION_RANDOM_SEED = 1701
CORRELATION_EXCLUDED_FRAGMENTS = (
    "timestamp",
    "datetime",
    "airlineid",
    "aircraftid",
    "engineid",
    "operatorid",
    "serialnumber",
    "assetidentifier",
    "calendarid",
)

# Quote arbitrary Spark column names so punctuation cannot be interpreted as nested-field syntax.
def _quoted_spark_column(name):
    return F.col(f"`{name.replace('`', '``')}`")


def _eligible_correlation_parameter(field):
    normalized = field.name.lower().replace("_", "")
    is_parameter = field.name.startswith("Parameter:")
    is_numeric = isinstance(field.dataType, (NumericType, BooleanType))
    is_metadata = any(fragment in normalized for fragment in CORRELATION_EXCLUDED_FRAGMENTS)
    return is_parameter and is_numeric and not is_metadata


# Correlation is a screening measure, not evidence that one engine signal causes another.
correlation_candidate_columns = [
    field.name for field in cda.schema.fields if _eligible_correlation_parameter(field)
]
correlation_candidate_set = set(correlation_candidate_columns)

top_results_widget = widgets.BoundedIntText(
    value=50,
    min=5,
    max=200,
    step=5,
    description="Top results:",
    layout=widgets.Layout(width="260px"),
)
row_limit_widget = widgets.BoundedIntText(
    value=50000,
    min=1000,
    max=500000,
    step=10000,
    description="Max rows:",
    layout=widgets.Layout(width="260px"),
)
minimum_pairs_widget = widgets.BoundedIntText(
    value=30,
    min=3,
    max=10000,
    step=10,
    description="Minimum pairs:",
    layout=widgets.Layout(width="260px"),
)
run_correlation_btn = widgets.Button(
    description="Run correlation analysis",
    button_style="primary",
    layout=widgets.Layout(width="240px"),
)
correlation_output = widgets.Output()


def _format_ranked_correlations(records, top_count):
    """Build a wide, scrollable table whose signed values cannot be hidden by truncation."""
    positive = sorted(
        (record for record in records if record[1] > 0),
        key=lambda record: (-record[1], record[0]),
    )[:top_count]
    negative = sorted(
        (record for record in records if record[1] < 0),
        key=lambda record: (record[1], record[0]),
    )[:top_count]
    output_length = max(len(positive), len(negative), 1)

    def _cell(record, value_class):
        if record is None:
            return '<td class="correlation-empty">No qualifying correlation</td>'
        parameter, value, _ = record
        visible_name = html.escape(_short_parameter_name(parameter))
        signed_value = html.escape(f"{value:+.6f}")
        return (
            '<td><div class="correlation-entry">'
            f'<span class="correlation-name">{visible_name}</span>'
            f'<span class="correlation-value {value_class}">{signed_value}</span>'
            "</div></td>"
        )

    rows = []
    for index in range(output_length):
        positive_record = positive[index] if index < len(positive) else None
        negative_record = negative[index] if index < len(negative) else None
        rows.append(
            "<tr>"
            + _cell(positive_record, "positive-value")
            + _cell(negative_record, "negative-value")
            + "</tr>"
        )

    return """
    <style>
      .correlation-scroll {
        width: 100%; max-height: 650px; overflow: auto;
        border: 1px solid #c8c8c8; border-radius: 4px; background: #ffffff;
      }
      .correlation-table {
        width: 100%; min-width: 1100px; table-layout: fixed;
        border-collapse: collapse; font-size: 13px;
      }
      .correlation-table th {
        position: sticky; top: 0; z-index: 2; padding: 10px 12px;
        border-bottom: 2px solid #9a9a9a; background: #f5f5f5; text-align: left;
      }
      .correlation-table th:first-child { color: #176b2c; }
      .correlation-table th:last-child { color: #a11a1a; }
      .correlation-table td {
        width: 50%; padding: 7px 12px; vertical-align: top;
        border-bottom: 1px solid #e6e6e6;
      }
      .correlation-table td:first-child { border-right: 1px solid #dddddd; }
      .correlation-entry { display: flex; align-items: flex-start; gap: 14px; }
      .correlation-name {
        flex: 1 1 auto; min-width: 0; overflow-wrap: anywhere; word-break: break-word;
        color: #222222; line-height: 1.35;
      }
      .correlation-value {
        flex: 0 0 88px; text-align: right; white-space: nowrap;
        font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
        font-weight: 800; font-size: 14px;
      }
      .positive-value { color: #137333; }
      .negative-value { color: #b3261e; }
      .correlation-empty { color: #777777; font-style: italic; }
    </style>
    <div class="correlation-scroll">
      <table class="correlation-table">
        <thead><tr>
          <th>Positive correlations (highest to lowest)</th>
          <th>Negative correlations (lowest to highest)</th>
        </tr></thead>
        <tbody>
    """ + "".join(rows) + """
        </tbody>
      </table>
    </div>
    """


def _run_correlation_analysis(_):
    run_correlation_btn.disabled = True
    try:
        with correlation_output:
            clear_output(wait=True)

            input_columns = []
            for short_name in selected_parameters:
                full_name = _full_parameter_name(short_name)
                if full_name in correlation_candidate_set and full_name not in input_columns:
                    input_columns.append(full_name)

            if not input_columns:
                print("None of the applied input features are eligible numeric or Boolean parameters.")
                return
            if len(input_columns) > CORRELATION_MAX_INPUTS:
                print(
                    f"Correlation is limited to {CORRELATION_MAX_INPUTS} input features per run. "
                    "Reduce the applied inputs in the parameter-selection cell."
                )
                return

            source_row_count = cda.count()
            if source_row_count == 0:
                print("No flight samples are available for correlation analysis.")
                return

            row_limit = int(row_limit_widget.value)
            sample_fraction = min(1.0, row_limit / source_row_count)
            minimum_pairs = int(minimum_pairs_widget.value)
            top_count = int(top_results_widget.value)
            correlations_by_input = {column: [] for column in input_columns}

            print(
                f"Inputs: {len(input_columns)} | Candidate parameters: "
                f"{len(correlation_candidate_columns):,} | Flight rows: {source_row_count:,}"
            )
            if sample_fraction < 1.0:
                print(
                    f"Using a reproducible random sample of approximately {row_limit:,} rows "
                    "to bound runtime and memory use."
                )

            for batch_start in range(0, len(correlation_candidate_columns), CORRELATION_BATCH_SIZE):
                batch_columns = correlation_candidate_columns[
                    batch_start:batch_start + CORRELATION_BATCH_SIZE
                ]
                required_columns = list(dict.fromkeys(input_columns + batch_columns))
                batch_df = cda.select(
                    *[_quoted_spark_column(column).alias(column) for column in required_columns]
                )
                if sample_fraction < 1.0:
                    batch_df = batch_df.sample(
                        withReplacement=False,
                        fraction=sample_fraction,
                        seed=CORRELATION_RANDOM_SEED,
                    ).limit(row_limit)

                aggregate_expressions = []
                result_keys = []
                for input_index, input_column in enumerate(input_columns):
                    left_raw = _quoted_spark_column(input_column).cast("double")
                    left = F.when(left_raw.isNull() | F.isnan(left_raw), None).otherwise(left_raw)
                    for candidate_index, candidate_column in enumerate(batch_columns):
                        if candidate_column == input_column:
                            continue
                        right_raw = _quoted_spark_column(candidate_column).cast("double")
                        right = F.when(right_raw.isNull() | F.isnan(right_raw), None).otherwise(right_raw)
                        correlation_alias = f"correlation_{input_index}_{candidate_index}"
                        overlap_alias = f"overlap_{input_index}_{candidate_index}"
                        aggregate_expressions.extend(
                            [
                                F.corr(left, right).alias(correlation_alias),
                                F.count(F.when(left.isNotNull() & right.isNotNull(), 1)).alias(overlap_alias),
                            ]
                        )
                        result_keys.append(
                            (
                                input_column,
                                candidate_column,
                                correlation_alias,
                                overlap_alias,
                            )
                        )

                if aggregate_expressions:
                    aggregate_row = batch_df.agg(*aggregate_expressions).first().asDict()
                    for input_column, candidate_column, correlation_alias, overlap_alias in result_keys:
                        correlation_value = aggregate_row.get(correlation_alias)
                        overlap_count = int(aggregate_row.get(overlap_alias) or 0)
                        if (
                            correlation_value is not None
                            and math.isfinite(float(correlation_value))
                            and overlap_count >= minimum_pairs
                        ):
                            correlations_by_input[input_column].append(
                                (candidate_column, float(correlation_value), overlap_count)
                            )

                completed = min(
                    batch_start + CORRELATION_BATCH_SIZE,
                    len(correlation_candidate_columns),
                )
                print(f"Processed {completed:,} of {len(correlation_candidate_columns):,} candidates.")

            for input_column in input_columns:
                ipy_display(
                    HTML(
                        "<h3 style='margin:18px 0 4px'>Input feature: "
                        + html.escape(_short_parameter_name(input_column))
                        + "</h3>"
                        + f"<p style='margin:0 0 8px'>Pearson correlations with at least "
                        f"{minimum_pairs:,} paired observations. Self-correlation and exact zero "
                        "correlations are omitted.</p>"
                    )
                )
                ranked_html = _format_ranked_correlations(
                    correlations_by_input[input_column],
                    top_count,
                )
                ipy_display(HTML(ranked_html))
    finally:
        run_correlation_btn.disabled = False


run_correlation_btn.on_click(_run_correlation_analysis)
correlation_controls = widgets.HBox(
    [top_results_widget, row_limit_widget, minimum_pairs_widget, run_correlation_btn]
)
ipy_display(widgets.VBox([correlation_controls, correlation_output]))


# 7. AC/BC schema coverage

**Purpose:** Verify whether every requested thrust-reverser suffix has both an AC and a BC column in the current scan schema.

**Output:** One row per channel-neutral signal showing AC/BC availability and whether each channel is in the current applied input selection. Unequal availability counts identify schema coverage differences, not signal-value disagreements.


In [ ]:
# Verify that every requested thrust-reverser suffix has matching AC and BC schema coverage.
schema_columns = set(cda_cols)
selected_parameter_set = set(selected_parameters)
channel_pair_records = []

for suffix in THRUST_REVERSER_SUFFIXES:
    ac_short = f"AC_{suffix}"
    bc_short = f"BC_{suffix}"
    ac_full = PARAMETER_PREFIX + ac_short
    bc_full = PARAMETER_PREFIX + bc_short
    channel_pair_records.append(
        {
            "Signal": suffix,
            "AC available": ac_full in schema_columns,
            "BC available": bc_full in schema_columns,
            "AC selected": ac_short in selected_parameter_set,
            "BC selected": bc_short in selected_parameter_set,
        }
    )

channel_pair_df = pd.DataFrame(channel_pair_records)
unpaired_schema_rows = channel_pair_df.loc[
    channel_pair_df["AC available"] != channel_pair_df["BC available"]
]

print(f"Thrust-reverser signal pairs: {len(channel_pair_df)}")
print(f"Schema pairs with unequal AC/BC availability: {len(unpaired_schema_rows)}")
ipy_display(channel_pair_df)

# 8. Flight master inventory

**Purpose:** Display the read-only master records for the active aircraft, engine, and calendar range.

**Output:** Available identity, timing, operator, audit, and calendar fields, ordered from the latest flight start. Duration is derived from master start and end timestamps.


In [ ]:
# Keep the detailed flight inventory read-only and tied to the active widget selections.
diagnostic_master_columns = [
    "AircraftIdentifier",
    "EngineSerialNumber",
    "AircraftId",
    "EngineId",
    "StartDatetime",
    "EndDatetime",
    "Duration",
    "EnginePosition",
    "OperatorId",
    "OperatorCode",
    "LastGeneratedDatetime",
    "FirstGeneratedDatetime",
    "Changed",
    "Created",
    "Migrated",
    "CalendarId",
]

diagnostic_master_with_duration = cda_master_dt.withColumn(
    "Duration",
    F.col("EndDatetime") - F.col("StartDatetime"),
)
diagnostic_available_columns = [
    column
    for column in diagnostic_master_columns
    if column in diagnostic_master_with_duration.columns
]
diagnostic_master_df = (
    diagnostic_master_with_duration.select(*diagnostic_available_columns)
    .orderBy(F.col("StartDatetime").desc(), F.col("EndDatetime").desc())
)

print(
    f"Flight inventory for aircraft {selected_acid}, engine {selected_esn}, "
    f"from {window_start_date} through {window_end_date}"
)
display(diagnostic_master_df)

# 9. Normalized selected flight windows

**Purpose:** Show the exact flight intervals used to filter the scan data.

**Output:** One normalized row per start time, its maximum valid end time, and the derived duration, ordered chronologically. This table is the clearest audit of which flights feed the plots and correlations.


In [ ]:
# Show every normalized flight window included in the active multi-day selection.
if flight_available:
    diagnostic_selected_master_df = (
        flight_windows_df.withColumn(
            "Duration",
            F.col("EndDatetime") - F.col("StartDatetime"),
        )
        .orderBy(F.col("StartDatetime"))
    )
    print(
        f"Selected flight windows: {len(flight_rows)} from "
        f"{window_start_date} through {window_end_date}"
    )
else:
    diagnostic_selected_master_df = flight_windows_df.limit(0)
    print("No selected flight windows are available.")

display(diagnostic_selected_master_df)

# 10. AC/BC population and exact disagreement

**Purpose:** Count non-null observations for paired AC/BC thrust-reverser signals and measure how often both populated values are not exactly equal.

**Output:** Per-signal AC population, BC population, overlapping population, and exact disagreement count.

**Caution:** Exact comparison is useful for discrete values. Continuous analog measurements may need an engineering tolerance before a disagreement is operationally meaningful.


In [ ]:
# Quantify AC/BC population and exact disagreement without assuming an engineering tolerance.
channel_coverage_records = []
for suffix in THRUST_REVERSER_SUFFIXES:
    ac_column = f"{PARAMETER_PREFIX}AC_{suffix}"
    bc_column = f"{PARAMETER_PREFIX}BC_{suffix}"
    if ac_column not in _data_df.columns or bc_column not in _data_df.columns:
        continue

    ac_values = _data_df[ac_column]
    bc_values = _data_df[bc_column]
    both_present = ac_values.notna() & bc_values.notna()
    exact_disagreement = (
        ac_values.loc[both_present].ne(bc_values.loc[both_present]).fillna(False)
    )
    channel_coverage_records.append(
        {
            "Signal": suffix,
            "AC non-null": int(ac_values.notna().sum()),
            "BC non-null": int(bc_values.notna().sum()),
            "Both non-null": int(both_present.sum()),
            "Exact disagreements": int(exact_disagreement.sum()),
        }
    )

channel_coverage_df = pd.DataFrame(channel_coverage_records)
if channel_coverage_df.empty:
    print("No paired AC/BC samples are available for coverage analysis.")
else:
    print(f"Channel coverage calculated from {len(_data_df):,} flight samples.")
ipy_display(channel_coverage_df)

# 11. Ordered sample preview

**Purpose:** Provide a compact, deterministic inspection of the active dataset after all selections have been applied.

**Output:** The first 50 timestamp-ordered rows containing identifiers, numeric `FlightPhase`, and the currently applied signal columns. This is a preview only; it does not alter any notebook state.


In [ ]:
# Preview the active flight deterministically without overwriting notebook-wide identifiers.
preview_identifier_columns = [
    column
    for column in IDENTIFIER_COLUMNS
    if column in select_para_df.columns
]
preview_parameter_columns = [
    _full_parameter_name(short_name)
    for short_name in selected_parameters
    if _full_parameter_name(short_name) in select_para_df.columns
]
preview_columns = preview_identifier_columns + [
    column
    for column in preview_parameter_columns
    if column not in preview_identifier_columns
]

diagnostic_preview_df = (
    select_para_df.select(*preview_columns)
    .orderBy(F.col("Timestamp"))
    .limit(50)
)
print(
    f"First 50 ordered samples for aircraft {selected_acid}, engine {selected_esn}, "
    f"{len(flight_rows)} flights from {window_start_date} through {window_end_date}"
)
display(diagnostic_preview_df)